# is_lotte_related 이진 분류기 학습 (Google Colab)

lotte-insight 프로젝트 — `training/train_lotte_related_classifier.py` Colab 실행용 노트북

**모델:** `monologg/koelectra-small-v3-discriminator` fine-tuning  
**분류 방식:** 이진 분류 (BCEWithLogitsLoss, num_labels=1)  
**목표:** val recall ≥ 0.97, precision ≥ 0.90  
**예상 소요:** T4 GPU 기준 약 5분

**사전 준비**
- 런타임 유형: T4 GPU (런타임 → 런타임 유형 변경)
- 업로드할 파일: `training/data/labeled_titles.csv`, `training/data/labeled_players.csv`

In [ ]:
# 1. GPU 확인
import torch
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA:', torch.version.cuda)
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('[WARN] GPU 없음 — CPU 학습 시 약 1.5~2.5시간 소요')

In [ ]:
# 2. 레포 클론 및 의존성 설치
GITHUB_REPO_URL = 'https://github.com/JoeYunHa/Lotte_Insight.git'

!git clone {GITHUB_REPO_URL} /content/lotte-insight
%cd /content/lotte-insight/training
!pip install -q transformers torch scikit-learn pandas numpy

In [ ]:
# 3. 학습 데이터 업로드 (로컬 → Colab)
# 실행 후 파일 선택 창에서 아래 두 파일을 선택하시오:
#   - labeled_titles.csv
#   - labeled_players.csv
import os
from google.colab import files

DATA_DIR = '/content/lotte-insight/training/data'
os.makedirs(DATA_DIR, exist_ok=True)

uploaded = files.upload()  # 파일 선택 창 열림

for fname, content in uploaded.items():
    dst = f'{DATA_DIR}/{os.path.basename(fname)}'
    with open(dst, 'wb') as f:
        f.write(content)
    print(f'저장 완료: {dst}  ({len(content):,} bytes)')

In [ ]:
# 4. 학습 데이터 분포 확인
import pandas as pd

total_pos = 0
total_neg = 0
total_null = 0

for fname in ['labeled_titles.csv', 'labeled_players.csv']:
    path = f'{DATA_DIR}/{fname}'
    if not os.path.exists(path):
        print(f'[MISSING] {fname}')
        continue
    df = pd.read_csv(path, encoding='utf-8-sig')
    if 'is_lotte_related' not in df.columns:
        print(f'[SKIP] {fname}: is_lotte_related 컬럼 없음')
        continue
    null_count = df['is_lotte_related'].isna().sum()
    labeled = df.dropna(subset=['is_lotte_related'])
    raw = labeled['is_lotte_related'].astype(str).str.strip().str.lower()
    pos = (raw.isin({'true', '1', 'yes'})).sum()
    neg = (raw.isin({'false', '0', 'no'})).sum()
    invalid = len(raw) - pos - neg
    total_pos += pos
    total_neg += neg
    total_null += null_count
    print(f'{fname}: 총 {len(df)}행  True={pos}  False={neg}  null={null_count}  invalid={invalid}')

print(f'\n합산 — True: {total_pos}  False: {total_neg}  (null 제외)')
if total_pos + total_neg > 0:
    ratio = total_pos / (total_pos + total_neg)
    print(f'pos 비율: {ratio:.2%}  (pos_weight 자동 계산됨)')
if total_pos < 200:
    print('[주의] True 샘플이 200건 미만 — recall 목표 달성 불안정 가능')

In [ ]:
# 5. 학습 실행
# best checkpoint: recall-first threshold 기준 자동 선택 및 저장
# T4 GPU 기준 약 5분 소요
!python train_lotte_related_classifier.py \
    --data-dir /content/lotte-insight/training/data \
    --output-dir /content/lotte-insight/training/models/lotte_related_koelectra \
    --epochs 5 \
    --lr 5e-5 \
    --batch 16

In [ ]:
# 6. 학습 결과 확인
import json, os

MODEL_DIR = '/content/lotte-insight/training/models/lotte_related_koelectra'
threshold_path = f'{MODEL_DIR}/threshold.json'

if os.path.exists(threshold_path):
    with open(threshold_path) as f:
        t = json.load(f)
    print(f'저장된 threshold: {t["threshold"]}')
    print('→ backend/models/lotte_related_detector.py가 이 값을 자동으로 로드합니다.')
else:
    print('[ERROR] threshold.json 없음 — 학습 실패 여부 확인 필요')

print('\n모델 파일 목록:')
for f in sorted(os.listdir(MODEL_DIR)):
    size = os.path.getsize(f'{MODEL_DIR}/{f}')
    print(f'  {f:<40} {size:>10,} bytes')

In [ ]:
# 7. 추론 테스트 (smoke test)
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_DIR = '/content/lotte-insight/training/models/lotte_related_koelectra'
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.eval()

with open(f'{MODEL_DIR}/threshold.json') as f:
    THRESHOLD = json.load(f)['threshold']

def predict(title: str, snippet: str = '') -> dict:
    enc = tokenizer(
        title, snippet[:300].strip(),
        truncation='only_second', padding='max_length',
        max_length=128, return_tensors='pt',
    )
    with torch.no_grad():
        logit = model(**enc).logits[0][0]
        prob = float(torch.sigmoid(logit))
    return {'is_lotte_related': prob >= THRESHOLD, 'prob': round(prob, 4)}

# 정답: True 케이스
TRUE_CASES = [
    ('롯데 나균안, 시즌 5승…선발 로테이션 안정화', '나균안이 두산전 6이닝 2실점 호투로 시즌 5승을 따냈다.'),
    ('롯데 전준우 햄스트링 부상, 2주 결장', '전준우가 1군 엔트리에서 말소됐다.'),
    ('사직구장 개막전 팬 3만 명 몰려', '롯데 자이언츠가 홈 개막전 이벤트를 성황리에 개최했다.'),
    ('롯데, 외국인 투수 교체 결정', '롯데가 부진한 외국인 투수를 방출하고 새 용병을 물색 중이다.'),
    ('서튼 감독 "선수들 잘 따라줬다"', '래리 서튼 감독이 경기 후 선수단을 칭찬했다.'),
]

# 정답: False 케이스
FALSE_CASES = [
    ('롯데백화점, 봄 세일 시작', '롯데백화점이 봄맞이 대규모 할인 행사를 시작했다.'),
    ('롯데월드, 신규 어트랙션 공개', '롯데월드가 여름 시즌 신규 놀이기구를 선보였다.'),
    ('삼성 라이온즈, KIA 잡고 선두 탈환', 'KIA 타이거즈가 삼성에 패해 2위로 내려앉았다.'),
]

print(f'Threshold: {THRESHOLD}\n')
print('=== True 케이스 (모두 True여야 정상) ===')
ok_true = 0
for title, snippet in TRUE_CASES:
    result = predict(title, snippet)
    mark = '✓' if result['is_lotte_related'] else '✗'
    print(f'  {mark} prob={result["prob"]:.4f}  {title}')
    if result['is_lotte_related']:
        ok_true += 1

print(f'\n=== False 케이스 (모두 False여야 정상) ===')
ok_false = 0
for title, snippet in FALSE_CASES:
    result = predict(title, snippet)
    mark = '✓' if not result['is_lotte_related'] else '✗'
    print(f'  {mark} prob={result["prob"]:.4f}  {title}')
    if not result['is_lotte_related']:
        ok_false += 1

total = len(TRUE_CASES) + len(FALSE_CASES)
passed = ok_true + ok_false
print(f'\n결과: {passed}/{total} 통과')
if passed < total:
    print('[주의] 실패 케이스 존재 — 셀 9에서 추가 학습 고려')

In [ ]:
# 8. 모델 다운로드 (Colab → 로컬)
# 압축 후 자동 다운로드 — 브라우저 팝업 차단 해제 필요 시 있음
import shutil
from google.colab import files

MODEL_DIR = '/content/lotte-insight/training/models/lotte_related_koelectra'
ZIP_PATH = '/content/lotte_related_koelectra.zip'

if os.path.exists(MODEL_DIR):
    shutil.make_archive('/content/lotte_related_koelectra', 'zip', MODEL_DIR)
    print(f'압축 완료: {ZIP_PATH}')
    print('압축 파일에 포함된 파일:')
    import zipfile
    with zipfile.ZipFile(ZIP_PATH) as z:
        for name in sorted(z.namelist()):
            info = z.getinfo(name)
            print(f'  {name:<45} {info.file_size:>10,} bytes')
    files.download(ZIP_PATH)
else:
    print('[ERROR] 모델 디렉토리 없음 — 학습 실패 여부 확인 필요')

## 다운로드 후 로컬 배치

```
lotte_related_koelectra.zip 압축 해제
  → training/models/lotte_related_koelectra/
```

필수 파일 확인:
- `config.json`
- `pytorch_model.bin` 또는 `model.safetensors`
- `tokenizer_config.json`
- `vocab.txt`
- `threshold.json`  ← 없으면 config의 IS_LOTTE_RELATED_THRESHOLD(0.40) 사용

배치 완료 후 `backend/models/lotte_related_detector.py`의 `_runtime`이 자동으로 모델을 탐색하여 하이브리드 게이트가 활성화됩니다.

In [ ]:
# 9. (선택) recall 미달 시 추가 학습
# 위 테스트에서 True 케이스 실패가 있거나 학습 로그에서 recall < 0.97이면 실행
#
# !python train_lotte_related_classifier.py \
#     --data-dir /content/lotte-insight/training/data \
#     --output-dir /content/lotte-insight/training/models/lotte_related_koelectra \
#     --epochs 8 \
#     --lr 3e-5 \
#     --batch 16
print('필요 시 위 주석을 해제하고 실행하세요.')